# Swap spread and futures positioning (EDGE-E5)

Does an OLS forecast from the change in leveraged-fund positioning in 10Y note futures (CFTC Traders in Financial Futures) and last week's spread change predict next week's 10Y SOFR OIS minus Treasury spread change? Pre-registration `research/prereg/E5.yaml`. The DTCC files before 2024-10-29 are in S3 Deep Archive, so the registered 2024 fitting period has only a few weeks; the run is shown as registered and the verdict is inconclusive.

Every number below is recomputed from the committed fixtures and checked against the ledger row; the plain-language finding is in `docs/research/EDGE_STUDIES.md`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from ratesresearch import backtest_analysis as ba
from ratesresearch.edge import report

ROOT = Path.cwd()
while not (ROOT / "data" / "presets.yaml").is_file():
    ROOT = ROOT.parent


def show(df: pd.DataFrame, digits: int = 4) -> None:
    display(Markdown(ba.to_markdown(df.round(digits))))

In [ ]:
from ratesresearch.edge import e5_swapspread

res = e5_swapspread.run(ROOT)
row = report.ledger_row(ROOT, "EDGE-E5")
display(Markdown(report.check_against_ledger(res, row)))
show(report.tests_table(res, row))

Band points and periods (unadjusted p; only the central holdout row is a registered test).

In [ ]:
show(
    pd.DataFrame(
        [
            {
                "band": b,
                "fitting weeks": x["fitting_weeks"],
                "holdout weeks": x["holdout_weeks"],
                "oos R2": x["oos_r2"],
                "mean u": x["test"]["mean"],
                "95% low": x["test"]["ci_low"],
                "95% high": x["test"]["ci_high"],
                "corr x1": x["corr_x1_target"],
                "corr x2": x["corr_x2_target"],
                "target sd bp": x["target_sd_bp"],
            }
            for b, x in res["band"].items()
        ]
    )
)

In [ ]:
display(Markdown(res["deviation"]))

In [ ]:
show(res["weekly"].tail(10))